In [ ]:
!pip install openpyxl

In [ ]:
import os
import pandas as pd
import re

# --- CONFIGURATION ---
root_dir = "/media/arnout/Elements/Thesis/Screen crispri quality checked"
excel_path = "/media/arnout/Elements/Thesis/Screen crispri quality checked/MetaData_Bsubt_KD_screen.xlsx" 
output_file = "index.csv"

def get_treatment_map(plate_name):
    """Loads the specific sheet for a plate and handles A1 -> A01 padding."""
    try:
        # 1. Load the sheet matching the Plate name
        df_sheet = pd.read_excel(excel_path, sheet_name=plate_name)
        
        def extract_gene(text):
            if pd.isna(text): return "Unknown"
            match = re.search(r'Gene target:\s*([\w-]+)', str(text))
            return match.group(1) if match else "Unknown"

        df_sheet['Treatment'] = df_sheet['Strain_name'].apply(extract_gene)
        
        # Ensure 'Position_microscopy' is treated as a string for mapping
        return dict(zip(df_sheet['Position_microscopy'].astype(str), df_sheet['Treatment']))
    except Exception as e:
        print(f"Warning: Could not find sheet '{plate_name}' in Excel. Treatment will be 'Unknown'.")
        return {}

def normalize_well_id(well_name):
    """Converts 'A1' or 'A11' into 'A01' or 'A11' to match Excel formatting."""
    match = re.match(r"([A-Z])([0-9]+)", well_name)
    if match:
        letter, number = match.groups()
        return f"{letter}{int(number):02d}" # Padd with zero if single digit
    return well_name

# --- DATA PROCESSING ---
all_rows = []
file_pattern = re.compile(r'Sample_(?P<well>[A-Z][0-9]+)_XY(?P<site>[0-9]+)_C(?P<chan>[0-9]+)')

# Loop through Plate folders
for plate_folder in os.listdir(root_dir):
    if not plate_folder.startswith("PLATE"): continue
    
    # Load the treatment map for THIS specific plate/sheet
    treatment_map = get_treatment_map(plate_folder)
    plate_path = os.path.join(root_dir, plate_folder)
    
    for well_folder in os.listdir(plate_path):
        if "bad_Q" in well_folder or not well_folder.startswith("Sample_"):
            continue
            
        well_path = os.path.join(plate_path, well_folder)
        raw_well_id = well_folder.replace("Sample_", "") # e.g. "A1"
        excel_well_id = normalize_well_id(raw_well_id)    # e.g. "A01"
        
        sites = {}
        for filename in os.listdir(well_path):
            if filename == "masks" or not filename.endswith(".tiff"): continue
            
            match = file_pattern.search(filename)
            if match:
                site_num = match.group('site')
                channel_num = f"C{match.group('chan')}"
                if site_num not in sites: sites[site_num] = {}
                sites[site_num][channel_num] = f"{plate_folder}/{well_folder}/{filename}"

        for site_id, channels in sites.items():
            all_rows.append({
                "Metadata_Plate": plate_folder,
                "Metadata_Well": raw_well_id, # DeepProfiler usually prefers A1, but pathing relies on this
                "Metadata_Site": int(site_id),
                "Treatment": treatment_map.get(excel_well_id, "Unknown"),
                "Replicate": 1,
                **channels
            })

# Save output
df_final = pd.DataFrame(all_rows)
channel_cols = sorted([c for c in df_final.columns if c.startswith("C")], key=lambda x: int(x[1:]))
final_cols = ["Metadata_Plate", "Metadata_Well", "Metadata_Site", "Treatment", "Replicate"] + channel_cols
df_final[final_cols].to_csv(output_file, index=False)

print(f"Successfully generated {output_file} with Sheet-specific treatments and padded Well IDs.")